# Patent Prior-Art Retrieval: reproducible Colab run

Prior-art search over **14,008 granted US patents** (HUPD, CPC classes H01M batteries and G06N AI/ML, filed
2012–2018), evaluated against citation-derived relevance judgments (206 queries).

This notebook rebuilds everything from the committed corpus and reproduces the report's results:

1. Build the BM25, BM25F and dense-embedding (FAISS) indices
2. Run every retrieval method and fuse them
3. Evaluate: test-split comparison with significance tests, the temporal prior-art filter, and 5-fold
   cross-validation over all 206 queries
4. Try the search engine on your own query

**Runtime → Run all.** A GPU runtime (Runtime → Change runtime type → T4 GPU) makes the embedding and
reranking steps much faster; on CPU the whole notebook takes roughly 20–30 minutes.

Full write-up: [`report/report.md`](https://github.com/Mokshagnya-MR/Patent-Retrieval-System/blob/master/report/report.md).
This is a prior-art discovery tool; it does not assess patentability or novelty.

## 1. Setup

In [ ]:
import os, shutil, subprocess, sys, time
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Mokshagnya-MR/Patent-Retrieval-System.git"

if IN_COLAB:
    repo = Path("/content/Patent-Retrieval-System")
    if not repo.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(repo)], check=True)
    os.chdir(repo)
    # Everything else (pandas, scikit-learn, sentence-transformers, lightgbm, scipy) is preinstalled on Colab.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rank-bm25", "faiss-cpu", "pytrec-eval-terrier"],
                   check=True)
elif Path.cwd().name == "notebooks":
    os.chdir("..")  # running locally from the repo's notebooks/ folder
sys.path.insert(0, os.getcwd())

# The pipeline reads the corpus from data/raw/ (its normal build location); the repo ships a compressed copy.
for src, dst in [("data/corpus/patents_subset.parquet", "data/raw/patents_subset.parquet"),
                 ("data/corpus/raw_citations.parquet", "data/qrels/raw_citations.parquet")]:
    if not Path(dst).exists():
        Path(dst).parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(src, dst)

import torch
print("Working directory:", os.getcwd())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU runtime)")


def run(module, *args):
    """Run one pipeline step (python -m <module>) and show the tail of its output."""
    t0 = time.time()
    print(f"$ python -m {module} {' '.join(args)}")
    r = subprocess.run([sys.executable, "-m", module, *args], capture_output=True, text=True)
    lines = [l for l in r.stdout.splitlines() if l.strip()]
    print("\n".join(lines[-8:]))
    if r.returncode != 0:
        print(r.stderr[-3000:])
        raise RuntimeError(f"{module} failed")
    print(f"done in {time.time() - t0:.0f}s\n")

In [ ]:
import pandas as pd
corpus = pd.read_parquet("data/raw/patents_subset.parquet")
topics = pd.read_csv("data/qrels/topics.tsv", sep="\t", header=None, names=["query_id", "text"])
qrels = pd.read_csv("data/qrels/qrels.txt", sep=" ", header=None, names=["query_id", "iter", "doc_id", "rel"])
print(f"{len(corpus):,} patents | {len(topics)} queries | {len(qrels)} relevance judgments")
corpus[["patent_id", "title", "filing_date", "main_cpc_label"]].head()

## 2. Build the indices
- **BM25** over title + abstract + claims
- **BM25F**: one BM25 index per field, combined with weights (claims ×2)
- **Dense**: `BAAI/bge-small-en-v1.5` embeddings of title + abstract, in exact (Flat) and approximate (IVF) FAISS
  indices

In [ ]:
run("src.index.bm25_index")
run("src.index.bm25f_index")
run("src.index.vector_index")

## 3. Retrieval runs
Each topic is a patent (title + abstract) and its relevant documents are the in-corpus patents it cites. The
query patent itself is excluded from its own results. Hybrid fusion's BM25 weight α is tuned on the **dev**
split only (144 queries) and applied to the **test** split (62 queries).

In [ ]:
run("src.retrieve.baseline")      # BM25 and BM25F
run("src.retrieve.semantic")      # dense retrieval
run("src.retrieve.hybrid")        # reciprocal rank fusion (no tuning)
run("src.retrieve.alpha_sweep")   # weighted fusion: alpha chosen on dev, applied to test
pd.read_csv("results/alpha_sweep.csv").round(4)

In [ ]:
run("src.retrieve.rerank")        # cross-encoder reranking of the hybrid top-100 (test split)

## 4. Evaluation

### 4.1 Test-split comparison
Paired Wilcoxon signed-rank tests on per-query NDCG@10, each method against the row above it, with
Holm–Bonferroni correction for the five tests.

In [ ]:
run("src.eval.final_comparison")
pd.read_csv("results/final_comparison_table.csv").round(4)

### 4.2 Temporal prior-art filter
A patent filed *after* the query patent cannot be prior art for it, but plain retrieval ignores dates. This
re-scores every run after dropping later-filed results.

In [ ]:
run("src.eval.temporal_filter")
pd.read_csv("results/temporal_filter.csv")[
    ["Method", "later_filed_in_top10", "NDCG@10", "NDCG@10_filtered", "NDCG@10_rel_change", "p_value_holm"]
].round(4)

### 4.3 Cross-validation over all 206 queries
62 test queries are too few to separate methods that differ by a few NDCG points. Here α is chosen on four
folds and applied to the fifth, so every query is scored once as held-out.

In [ ]:
run("src.eval.cross_validation")
pd.read_csv("results/cv_hybrid.csv").round(4)

### 4.4 Optional: learning-to-rank (LambdaMART)
Uses citation-graph features, so it is reported separately from the main comparison (see report Section 4).
Takes several minutes.

In [ ]:
RUN_LTR = False  # set to True to train and evaluate the LambdaMART reranker
if RUN_LTR:
    run("src.retrieve.rerank_ltr")
    from IPython.display import Image, display
    display(Image("results/ltr_feature_importance.png"))

## 5. Try it
Runs the same search pipeline the project's API serves: retrieval, optional prior-art date cutoff,
near-duplicate grouping, and the best-matching claim passages. Edit `QUERY` and re-run the cell, or use
`similar_to()` with a patent number from the corpus.

In [ ]:
import html
from IPython.display import HTML, display
from src.api.main import search, similar
from src.api.schemas import SearchRequest


def show(resp):
    rows = []
    for r in resp.results:
        passage = html.escape(r.highlighted_passages[0][:300]) + "…" if r.highlighted_passages else ""
        family = f"<br><small>grouped near-duplicates: {', '.join(r.family_members)}</small>" if r.family_members else ""
        rows.append(f"<tr><td>{r.rank}</td><td><b>US {int(r.patent_id):,}</b><br><small>{r.cpc_class} · filed "
                    f"{r.filing_date}</small></td><td><b>{html.escape(r.title)}</b><br><small>{passage}</small>"
                    f"{family}</td><td>{r.score:.3f}</td></tr>")
    note = f"{len(resp.results)} results in {resp.latency_ms:.0f} ms ({resp.method})"
    if resp.filed_before:
        note += f"; hid {resp.n_excluded_later_filed} patents filed after {resp.filed_before}"
    display(HTML(f"<p>{note}</p><table><tr><th>#</th><th>Patent</th><th>Title / best-matching claim passage</th>"
                 f"<th>Score</th></tr>{''.join(rows)}</table>"))


def similar_to(patent_number, k=5):
    show(similar(str(patent_number), k=k, method="hybrid", prior_art_only=True, diversify=True))


QUERY = "sulfide solid electrolyte for an all-solid-state lithium battery"
show(search(SearchRequest(query=QUERY, k=5, method="hybrid")))

In [ ]:
similar_to(10521475)  # prior art for a patent in the corpus (only earlier filings are shown)